<a href="https://colab.research.google.com/github/Aryanupadhyay23/PyTorch/blob/master/RNN_using_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [134]:
import pandas as pd

df = pd.read_csv("/content/100_Unique_QA_Dataset.csv")
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [135]:
# tokenize
def tokenize(text):
  text = text.lower()
  text = text.replace("?","")
  text = text.replace("'","")
  return text.split()

In [136]:
# vocab
vocab = {"<UNK>":0}

In [137]:
def build_vocab(row):
  tokenized_question = tokenize(row["question"])
  tokenized_answer = tokenize(row["answer"])
  merged_tokens = tokenized_question + tokenized_answer
  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

In [138]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [139]:
len(vocab)

324

In [140]:
# convert word to numerical indices
def text_to_indices(text, vocab):
  indexed_text = []
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab["<UNK>"])
  return indexed_text

In [141]:
text_to_indices("what is your name?", vocab)

[1, 2, 0, 0]

In [142]:
import torch
from torch.utils.data import Dataset, DataLoader

In [143]:
class QADataset(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    numerical_question = text_to_indices(self.df.iloc[idx]["question"], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[idx]["answer"], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)


In [144]:
dataset = QADataset(df, vocab)

In [145]:
dataset[56]

(tensor([ 42,   2,   3, 210, 137, 168, 211, 169]), tensor([113]))

In [146]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [147]:
for question, answer in dataloader:
  print(question)
  print(answer)
  break

tensor([[ 42,  18,   2,   3, 281,  12,   3, 282]])
tensor([[205]])


In [148]:
import torch.nn as nn
import torch.optim as optim

In [149]:
class SimpleRNN(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, 324)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))
    return output

In [150]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1, 6)
print("shape of a:", a.shape)

b = x(a)
print("shape of b:", b.shape)

c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [151]:
learning_rate = 0.001
epochs = 20

In [152]:
model = SimpleRNN(len(vocab))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [153]:
# training loop

for epoch in range(epochs):
  total_loss = 0
  for question, answer in dataloader:
    optimizer.zero_grad()
    output = model(question)
    loss = loss_fn(output, answer[0])
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 525.270031
Epoch: 2, Loss: 459.940087
Epoch: 3, Loss: 383.570177
Epoch: 4, Loss: 316.742455
Epoch: 5, Loss: 264.162175
Epoch: 6, Loss: 214.475171
Epoch: 7, Loss: 170.344272
Epoch: 8, Loss: 133.271868
Epoch: 9, Loss: 100.887140
Epoch: 10, Loss: 76.903528
Epoch: 11, Loss: 58.727510
Epoch: 12, Loss: 46.119248
Epoch: 13, Loss: 36.435942
Epoch: 14, Loss: 29.354571
Epoch: 15, Loss: 24.038305
Epoch: 16, Loss: 19.975998
Epoch: 17, Loss: 16.859656
Epoch: 18, Loss: 14.225939
Epoch: 19, Loss: 12.261896
Epoch: 20, Loss: 10.651406


In [154]:
def predict(model, question, threshold=0.5):
  numerical_question = text_to_indices(question, vocab)
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)
  output = model(question_tensor)
  probabilities = torch.softmax(output, dim=1)
  value, index = torch.max(probabilities, dim=1)
  if value < threshold:
    print("i don't know")
  else:
    print(list(vocab.keys())[index])

In [155]:
predict(model, "What is the capital of Japan?")

tokyo
